# 01 · Generate OkTex Pipeline Data → Unity Catalog Delta

**Target:** `stable_classic_wg38i9_catalog.oneok_okt`  
**Warehouse:** Serverless Starter Warehouse JPao

> **Data policy:** All measurement quantities are **synthetic**. Pipeline geography is placed within the *public* OKT system-map bounding box (lat 33.36→36.95, lon −102.33→−97.50: West Texas Panhandle → north-central Oklahoma). **No ONEOK or customer data is used.**

This notebook generates three tables — `dim_meters`, `pipeline_segments`, `fact_daily_measurements` — and loads them into Delta, then verifies.

### 1. Generate the synthetic dataset (pure Python, seeded)

In [ ]:
from generate_data import build_meters, build_segments, build_measurements

meters = build_meters()
segments = build_segments()
measurements = build_measurements(meters)
print(f'meters={len(meters)}  route_vertices={len(segments)}  daily_measurements={len(measurements)}')
print('meter types:', sorted({m["meter_type"] for m in meters}))

meters=20  route_vertices=14  daily_measurements=1200
meter types: ['DELIVERY', 'INTERCONNECT', 'RECEIPT']


### 2. Inspect a few generated rows

In [ ]:
import json
print('--- dim_meters[0] ---')
print(json.dumps(meters[0], indent=2))
print('--- fact_daily_measurements[0] ---')
print(json.dumps(measurements[0], indent=2))

--- dim_meters[0] ---
{
  "meter_id": "OKT-001",
  "meter_name": "Levelland Receipt",
  "meter_type": "RECEIPT",
  "latitude": 33.59,
  "longitude": -102.3,
  "county": "Hockley",
  "state": "TX",
  "segment": "West Texas",
  "pipe_diameter_in": 24,
  "capacity_dth": 120000,
  "operator": "OkTex Pipeline Company, L.L.C. (TSP 80-002-2246)",
  "status": "ACTIVE",
  "commissioned_year": 2004
}
--- fact_daily_measurements[0] ---
{
  "flow_date": "2026-06-30",
  "meter_id": "OKT-001",
  "scheduled_dth": 73978,
  "actual_dth": 76199,
  "pressure_psig": 746.7,
  "temperature_f": 79.8,
  "variance_pct": 3.0
}


### 3. Create & load the Delta tables (idempotent `CREATE OR REPLACE` + insert)
Executed against the Databricks SQL warehouse via the Statement Execution API.

In [ ]:
import load_delta
load_delta.main()

# OkTex data build -> stable_classic_wg38i9_catalog.oneok_okt

## Step 1: dim_meters
inserted 20 meter rows
meter_type   | meters | total_capacity_dth
-------------+--------+-------------------
DELIVERY     | 8      | 432000            
INTERCONNECT | 5      | 530000            
RECEIPT      | 7      | 695000            

## Step 2: pipeline_segments (route polyline)
inserted 14 route vertices
seq | latitude | longitude | segment_name
----+----------+-----------+-------------
1   | 33.59    | -102.3    | West Texas  
2   | 33.58    | -101.855  | West Texas  
3   | 34.184   | -101.708  | West Texas  
4   | 34.535   | -101.76   | West Texas  
5   | 34.98    | -101.92   | Panhandle   

## Step 3: fact_daily_measurements
inserted 1200 daily measurement rows
## Step 4: verification
### row counts
table                   | rows
------------------------+-----
dim_meters              | 20  
fact_daily_measurements | 1200
pipeline_segments       | 14  

### date range
first_day  | last_day   | 